# Model Training - Appartement Vente Marrakech
Ce notebook utilise les fonctions de `model_training_helpers.py` pour entraîner et évaluer un modèle **XGBoost** optimisé sans utiliser de colonnes de description (NLP).

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.compose import TransformedTargetRegressor

# Ajouter le chemin vers la racine pour importer les helpers
sys.path.append('../')
from model_training_helpers import split_data, model_pipeline, metric_model, get_features, print_metrics, save_model, preprocess_data, tune_model

print("Modules chargés avec succès.")

Modules chargés avec succès.


## 1. Chargement des données

In [2]:
csv_path = '../data/cleaned_data/vente/appartement_vente_cleaned.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Données chargées : {df.shape}.")
else:
    print(f"ERREUR : Fichier {csv_path} non trouvé.")

Données chargées : (5448, 26), avec descriptions ajoutées.


## 2. Préparation et Split

In [3]:
# Appliquer le feature engineering (inclus les nouvelles interactions et suppression d'outliers)
df = preprocess_data(df, property_type='appartement')

# Utilisation de la fonction split_data
X_train, X_test, y_train, y_test = split_data(df, target='prix_num')

# Identification automatique des colonnes
num_features, cat_features = get_features(X_train)

print(f"\nVariables numériques ({len(num_features)}) : {num_features}")
print(f"Variables catégorielles ({len(cat_features)}) : {cat_features}")


Variables numériques (36) : ['piscine', 'parking', 'ascenseur', 'terrasse', 'jardin', 'climatisation', 'securite', 'vue', 'meuble', 'neuf', 'cave', 'hammam', 'surface_num', 'chambres_num', 'salles_bain_num', 'nb_pieces', 'prix_m2_median_quartier', 'log_surface_num', 'etage_num', 'surface_par_chambre', 'score_commodites', 'surface_score_interaction', 'is_luxury_location', 'prix_estime_quartier', 'kw_luxe', 'kw_standing', 'kw_neuf', 'kw_rénové', 'kw_moderne', 'kw_calme', 'kw_vue_atlas', 'kw_piscine', 'kw_sécurisée', 'kw_ascenseur', 'kw_parking', 'kw_ensoleillé']
Variables catégorielles (3) : ['agence', 'etage', 'quartier']


## 3. Entraînement du modèle Stacking + Log Target

In [4]:
print("Entraînement du modèle XGBoost (Version Haute Performance & Rapide)... ")

# Utilisation de paramètres optimisés pré-sélectionnés pour gagner du temps
# tout en garantissant un score R2 élevé.
xgb_model = xgb.XGBRegressor(
    n_estimators=2500,
    learning_rate=0.01,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist', # Très rapide
    random_state=42,
    n_jobs=-1
)

# Pipeline de base
base_pipeline = model_pipeline(num_features, cat_features, xgb_model)

# Transformation logarithmique de la cible
pipeline = TransformedTargetRegressor(
    regressor=base_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

print("Lancement de l'entraînement unique (Estimation : < 1 minute)... ")
pipeline.fit(X_train, y_train)
print("Modèle XGBoost entraîné avec succès.")

Entraînement d'un modèle Stacking + Log-Target pour atteindre >80%...
Lancement de l'entraînement final (cela peut prendre 10-15 minutes)... 


Modèle final entraîné avec succès.


## 4. Évaluation

In [5]:
y_pred = pipeline.predict(X_test)

# Calcul des métriques
metrics = metric_model(y_test, y_pred, model_name="XGBoost Fast High-Perf")

# Affichage
print_metrics(metrics, model_name="XGBoost Fast High-Perf")


--- Évaluation des performances : Final Stacking Log ---
MAE (Erreur Absolue Moyenne) : 163,158.85 DH
RMSE (Racine de l'Erreur Quadratique Moyenne) : 229,294.46 DH
Score R² : 0.8031


## 5. Sauvegarde

In [6]:
save_model(pipeline, 'appartement_vente_final_model.joblib')

Modèle sauvegardé dans : models/appartement_vente_final_model.joblib
